# Exploratory Data Analysis: ABS Building Approvals

This notebook explores the ABS 8731.0 quarterly dwelling approvals data.
Key questions:
- What is the seasonal pattern across LGAs?
- How does state-level variation look?
- Where is the 2022 structural break visible in the raw series?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

FEATURES_PATH = Path('../data/processed/features.parquet')

## 1. Load feature dataset

In [ ]:
df = pd.read_parquet(FEATURES_PATH)
df['quarter'] = pd.PeriodIndex(df['quarter'], freq='Q')
print(f"Shape: {df.shape}")
print(f"Date range: {df['quarter'].min()} to {df['quarter'].max()}")
print(f"LGAs: {df['lga_code'].nunique()}")
df.head()

## 2. National aggregate: quarterly approvals over time

In [ ]:
national = df.groupby('quarter')['dwellings_approved'].sum().reset_index()
national['quarter_dt'] = national['quarter'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(national['quarter_dt'], national['dwellings_approved'], color='steelblue')
ax.axvline(pd.Timestamp('2022-07-01'), color='red', linestyle='--', label='RBA rate hikes begin (Q3 2022)')
ax.set_title('National Quarterly Dwelling Approvals')
ax.set_ylabel('Total dwellings approved')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Seasonal decomposition

In [ ]:
seasonal = df.groupby(df['quarter'].apply(lambda q: q.quarter))['dwellings_approved'].mean()
seasonal.index = ['Q1', 'Q2', 'Q3', 'Q4']

fig, ax = plt.subplots(figsize=(8, 4))
seasonal.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Average Approvals by Quarter (seasonal pattern)')
ax.set_ylabel('Mean dwellings approved')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

## 4. Cash rate vs approvals: the 2022 structural break

In [ ]:
macro = df.drop_duplicates('quarter').sort_values('quarter')[['quarter', 'cash_rate', 'post_rate_hike']].copy()
macro['quarter_dt'] = macro['quarter'].dt.to_timestamp()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.plot(national['quarter_dt'], national['dwellings_approved'], color='steelblue', label='Approvals (left)')
ax2.plot(macro['quarter_dt'], macro['cash_rate'], color='darkorange', linestyle='--', label='Cash rate % (right)')

ax1.axvline(pd.Timestamp('2022-07-01'), color='red', linestyle=':', linewidth=2, label='Structural break')
ax1.set_title('Dwelling Approvals vs RBA Cash Rate')
ax1.set_ylabel('Dwellings approved', color='steelblue')
ax2.set_ylabel('Cash rate (%)', color='darkorange')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)
plt.tight_layout()
plt.show()

## 5. Class distribution: approvals across LGAs

In [ ]:
from data.pipeline import describe
describe(df)